## 0 · Instalación de dependencias

In [ ]:
# Instalar todas las dependencias necesarias para el pipeline.
# Se usa --quiet para reducir el output. Solo es necesario ejecutar esto
# una vez por entorno; en ejecuciones posteriores se puede saltar.
#
# Paquetes principales:xs
#   openeo         → cliente Python para el backend Copernicus Dataspace
#   xarray         → manejo de datos N-dimensionales (cubos de datos)
#   geopandas      → operaciones geoespaciales vectoriales
#   rioxarray      → extensión geoespacial para xarray (CRS, transform)
#   rasterio       → lectura/escritura de GeoTIFFs
#   contextily     → basemaps (imágenes de fondo satelital/OSM)
#   scikit-image   → remuestreo de rasters (resize con interpolación)
#   netcdf4        → motor para leer ficheros NetCDF descargados de OpenEO
%pip install openeo xarray geopandas rioxarray matplotlib contextily rasterio pyproj shapely scikit-image netcdf4 --quiet


## 1 · Imports y funciones auxiliares

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# IMPORTS — Librer\u00edas est\u00e1ndar y módulos del proyecto
# ════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd
import xarray as xr
import rasterio
import contextily as ctx
import openeo

# Módulos del proyecto
# utils.py contiene TODAS las funciones del pipeline (descarga, procesamiento, viz)
from utils import *

# intertidal_toolkit.py contiene clases modulares (opcional, para código más organizado)
from intertidal_toolkit import (
    CoordinateUtils,
    RasterUtils, 
    OpenEOManager,
    SCLProcessor,
    IntertidalMapper,
    TideAnalyzer,
    Visualizer
)

# Módulo de modelos de marea
from tidemodel import PyTMDTideModel, CopernicusTideModel

print("✓ Imports completados")
print("✓ utils.py: funciones del pipeline cargadas")
print("✓ intertidal_toolkit.py: clases modulares cargadas")
print("✓ tidemodel.py: modelos de marea cargados")

## 2 · Configuración

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN PRINCIPAL — modifica aquí para cambiar de sitio o periodo
# ════════════════════════════════════════════════════════════════════════════

# Nombre del sitio (aparece en títulos de gráficos y ficheros de salida)
site = "Foz"

# ── Intervalo temporal ────────────────────────────────────────────────────────
# Periodo de descarga de las escenas RGB/SCL de muestra (secciones 4-11).
# Se usa también para explorar qué fechas hay disponibles en el backend.
time_extent = ["2024-01-01", "2024-12-31"]

# ── Fechas de muestra a descargar ─────────────────────────────────────────────
# Subconjunto representativo para visualización y análisis exploratorio.
# El Reference Map (sección 12) usa un año completo de forma independiente.
# Tip: ejecuta primero la sección 3 para ver todas las fechas disponibles.
selected_dates = [
    "2024-01-04",   # Invierno
    "2024-03-14",   # Primavera temprana
    "2024-05-28",   # Primavera
    "2024-07-02",   # Verano
    "2024-08-11",   # Verano
    "2024-09-05",   # Finales verano
    "2024-10-30",   # Otoño
    "2024-12-09",   # Invierno
]

# ── Umbral de calidad SCL (filtro global) ─────────────────────────────────────
# Una escena se descarta si el porcentaje de píxeles "malos" supera este valor.
# Este es el filtro estándar (PASO 3). El filtro mejorado por zona de transición
# (PASO 5) puede recuperar escenas que se descartan aquí.
scl_bad_fraction = 0.20  # 20% → descarta si >20% del AOI está nublado/con sombras

# Clases SCL que se consideran "malas" (no aptas para análisis de agua/tierra):
#   3  = sombra de nube       → el clasificador detecta la sombra proyectada
#   8  = nube media prob.     → píxeles probablemente nubosos
#   9  = nube alta prob.      → píxeles muy probablemente nubosos
#  10  = cirrus               → nubes finas de gran altitud
#  11  = nieve/hielo          → puede confundirse con agua en zonas costeras
scl_bad_classes = [3, 8, 9, 10, 11]

# ── Directorios de salida ─────────────────────────────────────────────────────
dir_rgb = "tifs_rgb"   # GeoTIFFs RGB descargados (B04+B03+B02 = R+G+B)
dir_scl = "tifs_scl"   # GeoTIFFs SCL descargados (banda de clasificación)

# Crear las carpetas si no existen (exist_ok=True no falla si ya existen)
os.makedirs(dir_rgb, exist_ok=True)
os.makedirs(dir_scl, exist_ok=True)

# ── AOI — Área de Interés ─────────────────────────────────────────────────────
# Polígono de la Ría de Foz definido mediante coordenadas en formato DMS
# (grados°minutos'segundos"hemisferio).
#
# El polígono tiene 5 vértices en forma de pentágono para aproximar el estuario:
#   - Boca ancha al norte (costa atlántica)
#   - Forma convergente hacia el sur (cabecera del estuario)
aoi_dms = [
    '43°35\'24.00"N 7°17\'6.00"W',   # NW — boca, esquina noroeste
    '43°35\'24.00"N 7°12\'18.00"W',  # NE — boca, esquina noreste (playa de Foz)
    '43°32\'24.00"N 7°12\'18.00"W',  # SE — interior, esquina sureste
    '43°31\'12.00"N 7°14\'42.00"W',  # S  — cabecera del estuario (punto central)
    '43°32\'24.00"N 7°17\'6.00"W',   # SW — interior, esquina suroeste
]

# ── Usar CoordinateUtils para crear geometrías ───────────────────────────────
coords = CoordinateUtils()
polygon = coords.make_polygon(aoi_dms)
bbox = coords.bbox_from_polygon(polygon)

# Sistemas de referencia usados en el pipeline
crs = "EPSG:4326"           # WGS84 geográfico: coordenadas lon/lat en grados
crs_web_mercator = "EPSG:3857"  # Web Mercator: proyección plana en metros (para basemaps)

# ════════════════════════════════════════════════════════════════════════════
# VISUALIZACIÓN CARTOGRÁFICA DEL AOI
# ════════════════════════════════════════════════════════════════════════════
# Se generan dos paneles:
#   Izquierda: vista general de localización (zoom regional)
#   Derecha:   vista detallada del AOI sobre imagen satelital (zoom local)

fig, (ax_ov, ax_det) = plt.subplots(1, 2, figsize=(14, 6))

# Reproyectar el AOI a Web Mercator para poder usar contextily como basemap
# (contextily requiere que los datos estén en EPSG:3857)
aoi_wm = gpd.GeoDataFrame(
    geometry=[polygon], crs=crs
).to_crs(crs_web_mercator)

# ── Panel izquierdo: vista general (zoom regional) ───────────────────────────
b = aoi_wm.total_bounds  # [xmin, ymin, xmax, ymax] en Web Mercator
ax_ov.set_xlim(b[0], b[2])
ax_ov.set_ylim(b[1], b[3])
# zoom=10 → visión regional (~1:200.000), muestra la costa gallega
ctx.add_basemap(ax_ov, crs=crs_web_mercator, source=ctx.providers.Esri.WorldImagery, zoom=10)
aoi_wm.boundary.plot(ax=ax_ov, color="red", linewidth=2)  # Contorno del AOI en rojo
# Estrella roja en el centroide del AOI para identificarlo rápidamente
centroid = aoi_wm.geometry.iloc[0].centroid
ax_ov.plot(centroid.x, centroid.y, marker="*", color="red", markersize=14, zorder=5)
ax_ov.set_axis_off()

# ── Panel derecho: vista detallada (zoom local sobre el estuario) ─────────────
BUFFER = 2500  # Margen extra en metros alrededor del AOI para ver el contexto costero
b = aoi_wm.total_bounds
ax_det.set_xlim(b[0] - BUFFER, b[2] + BUFFER)
ax_det.set_ylim(b[1] - BUFFER, b[3] + BUFFER)
# zoom=13 → visión local (~1:25.000), se distinguen canales y marismas
ctx.add_basemap(ax_det, crs=crs_web_mercator, source=ctx.providers.Esri.WorldImagery, zoom=13)
aoi_wm.boundary.plot(ax=ax_det, color="red", linewidth=2, label="AOI")
ax_det.legend(loc="lower right", fontsize=9)
ax_det.set_axis_off()

fig.suptitle(f"Study Area: {site}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"AOI bbox: {bbox}")
print(f"Fechas seleccionadas: {len(selected_dates)}")

## 3 · Conexión a OpenEO y exploración de fechas disponibles

In [ ]:
# ── Conexión al backend de Copernicus Dataspace ───────────────────────────────
# Usamos OpenEOManager para gestionar la conexión de forma modular.
# También se mantiene 'conn' como variable directa para compatibilidad con
# funciones de utils.py que esperan una conexión openeo.

manager = OpenEOManager()
manager.connect()

# Extraer la conexión para usarla con funciones legacy de utils.py
conn = manager.connection

print("✓ OpenEOManager inicializado y conectado")
print(f"✓ Cuenta: {conn.describe_account()['name']}")

In [ ]:
# ── Exploración de fechas disponibles (PASO 1 del pipeline) ──────────────────
# Esta celda NO descarga ninguna imagen. Solo consulta al backend
# qué fechas de Sentinel-2 L2A están disponibles para el AOI y periodo.
#
# La operación dimension_labels("t") es muy barata (milisegundos) porque
# solo consulta el catálogo de metadatos, no los datos en sí.

# Definir el "proceso graph" de forma lazy (sin ejecutar todavía)
cube_explore = conn.load_collection(
    "SENTINEL2_L2A",            # Colección Sentinel-2 Nivel 2A (reflectancia superficial)
    spatial_extent=bbox,         # Recortar al bbox del AOI
    temporal_extent=time_extent, # Solo el periodo configurado
    bands=["B04"],               # Una sola banda basta para listar fechas (más barato)
    max_cloud_cover=100,         # Sin filtro de nubes aquí (se hace localmente con SCL)
)

# Ejecutar la consulta: devuelve la lista de fechas como strings ISO
all_dates = cube_explore.dimension_labels("t").execute()

# Mostrar el resultado con indicación de qué fechas se van a descargar
print(f"Fechas disponibles en el periodo {time_extent[0]} → {time_extent[1]}:")
print(f"  Total: {len(all_dates)}")
for d in all_dates:
    marker = " ← seleccionada" if d in selected_dates else ""
    print(f"  {d}{marker}")


## 4 · Descarga de imágenes RGB

Se lanza un batch job por cada fecha. Si el archivo ya existe, se salta (útil para retomar descargas interrumpidas).

In [ ]:
# ── Descarga de imágenes RGB (PASO 2a del pipeline) ──────────────────────────
# Para cada fecha en selected_dates se lanza un batch job en el backend
# de Copernicus que descarga las bandas B04 (rojo), B03 (verde), B02 (azul)
# recortadas al bbox del AOI.
#
# Cada job puede tardar 1-5 minutos. La descarga es SECUENCIAL para no saturar
# la cuota de jobs simultáneos del servicio gratuito.
#
# OPCIÓN 1: Usar OpenEOManager (enfoque OOP)
# OPCIÓN 2: Usar download_date_rgb() de utils.py (enfoque funcional)

print(f"Descargando {len(selected_dates)} fechas RGB...\n")

results_rgb = {}
for date in selected_dates:
    print(f"→ {date}")
    # Opción OOP: manager.download_date_rgb(date, bbox, dir_rgb)
    # Opción funcional (mantenemos por compatibilidad):
    results_rgb[date] = download_date_rgb(conn, date, bbox, dir_rgb)

# Resumen final
print("\n── Resumen RGB ──")
for date, status in results_rgb.items():
    print(f"  {date}: {status}")

## 5 · Descarga de SCL (Scene Classification Layer)

In [ ]:
# ── Descarga de imágenes SCL (PASO 2b del pipeline) ──────────────────────────
# Para las mismas fechas que tienen RGB descargado, se descarga la banda SCL
# (Scene Classification Layer). La SCL es un mapa 2D de 20 m/píxel donde
# cada píxel tiene una clase entera [0-11] que indica su tipo:
# agua, vegetación, suelo desnudo, nubes, sombras, nieve, etc.
#
# La detección se basa en las fechas que tienen RGB en DISCO (no en results_rgb
# del kernel) para ser robusta frente a reinicios del kernel o ejecuciones
# parciales del notebook.

# Encontrar las fechas que ya tienen RGB descargado en disco
dates_with_rgb = [
    d for d in selected_dates
    if os.path.exists(os.path.join(dir_rgb, f"rgb_{d}.tif"))
]

print(f"Descargando SCL para {len(dates_with_rgb)} fechas...\n")

results_scl = {}
for date in dates_with_rgb:
    print(f"→ {date}")
    # También idempotente: salta si scl_{date}.tif ya existe en tifs_scl/
    # Opción OOP: manager.download_date_scl(date, bbox, dir_scl)
    # Opción funcional (mantenemos por compatibilidad):
    results_scl[date] = download_date_scl(conn, date, bbox, dir_scl)

print("\n── Resumen SCL ──")
for date, status in results_scl.items():
    print(f"  {date}: {status}")

## 6 · Grid completo — todas las fechas descargadas

Se muestran todas las imágenes RGB. El título de cada panel incluye la fecha.

In [ ]:
# ── Grid completo de la serie temporal ───────────────────────────────────────
# Muestra todas las fechas descargadas en una cuadrícula para inspeccionarlas
# visualmente. Permite identificar rápidamente:
#   - Escenas nubladas o con sombras
#   - Variaciones estacionales de vegetación e inundación
#   - Posibles errores en la descarga (imagen negra o vacía)
#
# El polígono del AOI se usa como máscara: los píxeles fuera del pentágono
# se pintan en blanco para centrar la atención en el estuario.

# Listar las fechas que realmente tienen un fichero RGB en disco
# (puede ser un subconjunto de selected_dates si alguna descarga falló)
valid_rgb_dates = [
    d for d in selected_dates
    if os.path.exists(os.path.join(dir_rgb, f"rgb_{d}.tif"))
]

print(f"Mostrando {len(valid_rgb_dates)} imágenes")

# Mostrar en cuadrícula 4 columnas con normalización por percentiles (p2-p98)
# para maximizar el contraste visual de cada escena individual
plot_rgb_grid(
    dates=valid_rgb_dates,
    rgb_dir=dir_rgb,
    polygon=polygon,   # Máscara: oculta píxeles fuera del AOI
    title=f"Serie temporal completa — {site} (Sentinel-2 L2A, RGB)",
    cols=4,
)


## 7 · Cálculo del filtro SCL

Para cada escena se calcula qué fracción de sus píxeles cae en una clase "mala" (nubes, sombras, nieve). Las escenas por encima del umbral se descartan.

In [ ]:
# ── Filtro de calidad SCL — PASO 3 del pipeline ───────────────────────────────
# Para cada escena, se lee su SCL y se calcula qué fracción de los píxeles
# cae en una clase "mala" (nubes, sombras, etc.).
# Si esa fracción supera scl_bad_fraction → escena DESCARTADA.
#
# Este es el filtro GLOBAL estándar. Su limitación es que no distingue
# entre nubes sobre tierra firme (irrelevantes) y nubes sobre el estuario
# (las que realmente nos importan). El Reference Map (sección 12) mejora esto.

# Paleta de colores oficial de la ESA para visualizar mapas SCL
# Cada clave es el valor entero de la clase; el valor es (descripción, color_hex)
scl_colors = {
    0:  ("No data",             "#000000"),  # Sin datos
    1:  ("Saturado/Defectuoso", "#ff0000"),  # Píxel saturado o defectuoso
    2:  ("Sombra topográfica",  "#2f2f2f"),  # Sombra por terreno (no por nube)
    3:  ("Sombra de nube",      "#646464"),  # Sombra proyectada por una nube ← MALO
    4:  ("Vegetación",          "#00a000"),  # Superficie vegetal ← BUENO
    5:  ("No vegetación",       "#ffe65a"),  # Suelo desnudo, arena, roca ← BUENO
    6:  ("Agua",                "#0000ff"),  # Agua (mar, río, lago) ← BUENO
    7:  ("Sin clasificar",      "#808080"),  # No clasificado
    8:  ("Nube media prob.",    "#c0c0c0"),  # Nube de probabilidad media ← MALO
    9:  ("Nube alta prob.",     "#ffffff"),  # Nube de probabilidad alta ← MALO
    10: ("Cirrus",              "#87ceeb"),  # Nube de cirrus (alta altitud) ← MALO
    11: ("Nieve/Hielo",         "#e0ffff"),  # Nieve o hielo ← MALO en costas
}

# ── Calcular estadísticas SCL por escena ──────────────────────────────────────
scl_stats = {}  # Dict: {fecha: {'bad_fraction': float, 'bad_pct': float, 'class_counts': dict}}

for date in valid_rgb_dates:
    scl_path = os.path.join(dir_scl, f"scl_{date}.tif")

    # Leer el SCL de esta fecha como array 2D uint8
    scl_arr = tif_to_scl(scl_path)
    if scl_arr is None:
        print(f"    {date}: SCL no disponible")
        continue

    # Calcular % de píxeles malos (suma de píxeles en clases scl_bad_classes)
    stats = compute_scl_stats(scl_arr, scl_bad_classes)
    scl_stats[date] = stats

    # Mostrar resultado con veredicto
    verdict = "VÁLIDA" if stats["bad_fraction"] <= scl_bad_fraction else " DESCARTADA"
    print(f"  {date}  {stats['bad_pct']:5.1f}% malos  {verdict}")

# ── Clasificar fechas según el resultado del filtro ───────────────────────────
# Válidas: pasan el umbral → se usan en los análisis siguientes
dates_valid     = [d for d in valid_rgb_dates if d in scl_stats and scl_stats[d]["bad_fraction"] <= scl_bad_fraction]
# Descartadas: superan el umbral → se muestran en la sección 10 para inspección
dates_discarded = [d for d in valid_rgb_dates if d in scl_stats and scl_stats[d]["bad_fraction"] >  scl_bad_fraction]
# Sin SCL: el fichero no existe (descarga fallida o no lanzada)
dates_no_scl    = [d for d in valid_rgb_dates if d not in scl_stats]

print(f"\n── Resultado (umbral SCL = {scl_bad_fraction*100:.0f}%) ──")
print(f"  Válidas:    {len(dates_valid)}")
print(f"  Descartadas:{len(dates_discarded)}")
print(f"  Sin SCL:    {len(dates_no_scl)}")


## 9 · Grid de imágenes válidas

In [ ]:
# ── Grid solo con fechas válidas según SCL ─────────────────────────────────
plot_rgb_grid(
        dates=dates_valid,
        rgb_dir=dir_rgb,
        polygon=polygon,
        title=f"Escenas VÁLIDAS (≤{scl_bad_fraction*100:.0f}% píxeles malos) — {site}",
        cols=4,)

## 10 · Grid de imágenes descartadas

In [ ]:
# ── Grid solo con fechas descartadas según SCL ───────────────────────────────
if dates_discarded:
    subtitles_discarded = {
        d: f"{scl_stats[d]['bad_pct']:.1f}% malos"
        for d in dates_discarded if d in scl_stats
    }
    highlights_discarded = {d: "#e74c3c" for d in dates_discarded}

    plot_rgb_grid(
        dates=dates_discarded,
        rgb_dir=dir_rgb,
        polygon=polygon,
        title=f"Escenas DESCARTADAS (>{scl_bad_fraction*100:.0f}% píxeles malos) — {site}",
        cols=4,
    )
else:
    print("Todas las escenas pasaron el filtro SCL. 🎉")

## 11 · (Opcional) Visualizar el mapa SCL de una escena

In [ ]:
# ── Inspección visual de una escena SCL ──────────────────────────────────────
# Muestra el mapa SCL de una fecha concreta con los colores oficiales de la ESA.
# Útil para entender visualmente por qué una escena fue descartada o aceptada.
#
# Cambia el índice de selected_dates[] para inspeccionar otra fecha.
# Por ejemplo: [0] = primera fecha, [-1] = última fecha.

inspect_date = selected_dates[4]   # ← modifica este índice para cambiar la fecha

plot_scl_map(
    inspect_date,
    dir_scl,
    scl_colors=scl_colors,           # Paleta de colores ESA definida arriba
    scl_bad_classes=scl_bad_classes, # Clases a destacar como "malas"
    scl_stats=scl_stats,             # Estadísticas pre-calculadas para mostrar el %
    scl_max_bad_fraction=scl_bad_fraction,  # Umbral para mostrar VÁLIDA/DESCARTADA
)


In [ ]:
# ── Selección de fechas para el Reference Map local ──────────────────────────
# Para construir el reference map se necesitan escenas MUY limpias (< 5% de nubes).
# Se usa un umbral más estricto que el filtro general (5% vs 20%) porque el
# reference map es la "base de verdad" del análisis: errores aquí se propagan.
#
# NOTA: Este bloque usa el conjunto de muestra (selected_dates).
#       La construcción robusta vía OpenEO (año completo, sección 12)
#       es el enfoque recomendado para el análisis final.

reference_dates = [
    d
    for d, stats in scl_stats.items()
    if stats["bad_fraction"] <= 0.05  # Solo escenas con ≤5% de píxeles malos
]

print(f"Fechas seleccionadas para reference map: {len(reference_dates)}")
print(reference_dates)

# Cargar los SCLs de esas fechas y apilarlos en un array 3D (T, H, W)
# donde T = número de fechas, H = filas, W = columnas del raster
scl_stack = load_scl_stack(
    reference_dates,
    dir_scl,
)
print(f"\nShape del stack: {scl_stack.shape}  (fechas, filas, columnas)")


In [ ]:
# ── Construcción del Reference Map local (PASO 4 simplificado) ───────────────
# A partir del stack de SCLs limpios, se construye el mapa de estabilidad
# que clasifica cada píxel como:
#
#   0 → TRANSICIÓN (zona intermareal candidata)
#       El píxel no es estables: aparece como agua en algunas fechas y
#       como tierra en otras, O está en el buffer costero adyacente a la
#       zona de cambio.
#
#   1 → AGUA ESTABLE
#       El píxel NUNCA apareció como tierra en ninguna observación clara.
#       Está permanentemente bajo el agua (canal principal, mar abierto).
#
#   2 → TIERRA ESTABLE
#       El píxel NUNCA apareció como agua en ninguna observación clara.
#       Está permanentemente fuera del alcance de la marea.
#
# Parámetros clave:
#   stable_threshold=0.95      → necesita ≥95% de observaciones como agua/tierra
#                                 para clasificarse como estable
#   coastal_buffer_pixels=10   → añade 10 píxeles (= 200m a 20m/px) de buffer
#                                 alrededor del límite agua/tierra, capturando
#                                 píxeles fronterizos que podrían perderse

reference_map, P_water, P_land = build_reference_map(
    scl_stack,
    stable_threshold=0.95,
    coastal_buffer_pixels=10,
)

# La máscara de transición es simplemente los píxeles con valor 0 en el reference map
transition_mask = (reference_map == 0)  # True = zona intermareal

print(
    f"Píxeles en zona de transición: "
    f"{transition_mask.sum():,} "
    f"({100 * transition_mask.mean():.2f}% del AOI)"
)

# Visualizar el mapa con tres colores: rojo=transición, azul=agua, arena=tierra
plot_reference_map(reference_map)


In [ ]:
# ── Diagnóstico: comparar filtro global vs filtro en transición ───────────────
# Esta celda ilustra la diferencia entre los dos enfoques de filtrado:
#
#   FILTRO GLOBAL (PASO 3):
#     % nubes en todo el AOI > umbral → descarta la escena entera
#
#   FILTRO EN TRANSICIÓN (PASO 5):
#     % nubes dentro de la zona intermareal > umbral → descarta la escena
#     Una escena puede tener muchas nubes sobre tierra firme o mar abierto,
#     pero estar perfectamente despejada sobre el estuario → SE ACEPTA

# Ver las dimensiones del stack para confirmar que se cargó correctamente
print(f"Shape del SCL stack: {scl_stack.shape}  → ({scl_stack.shape[0]} fechas, {scl_stack.shape[1]}×{scl_stack.shape[2]} px)")

# ── Seleccionar una escena de prueba ──────────────────────────────────────────
test_date = "2024-07-02"

scl_path = os.path.join(dir_scl, f"scl_{test_date}.tif")
scl_original = tif_to_scl(scl_path)  # Array 2D uint8 con las clases SCL

# ── Calcular fracción de malos según el filtro GLOBAL ────────────────────────
bad_mask_original = np.isin(scl_original, scl_bad_classes)  # True donde hay nubes
bad_fraction_original = bad_mask_original.sum() / bad_mask_original.size
print(f"\nFiltro global  → {100*bad_fraction_original:.2f}% píxeles malos")

# ── Calcular fracción de malos según el filtro en TRANSICIÓN ─────────────────
transition_mask = (reference_map == 0)  # Solo los píxeles intermareales

# compute_transition_cloud_stats cuenta solo los píxeles malos que están
# DENTRO de la zona de transición, ignorando el resto del AOI
stats_transition = compute_transition_cloud_stats(
    scl_original,
    transition_mask,   # Máscara booleana 2D: True = zona intermareal
    scl_bad_classes,
)
print(f"Filtro transición → {stats_transition['bad_pct']:.2f}% nubes en zona intermareal")
print(f"\nConclusión: el filtro en transición puede aceptar escenas que el filtro")
print(f"global descartaría si las nubes están fuera del estuario.")


In [ ]:
# ── Visualización diagnóstica: 4 paneles comparativos ────────────────────────
# Muestra en una sola figura los 4 elementos clave para entender el filtrado:
#
#   Panel 1 — SCL original:    los valores brutos del clasificador
#   Panel 2 — Reference map:   las zonas de agua/tierra/transición estables
#   Panel 3 — Zona intermareal: la máscara donde se evalúan las nubes
#   Panel 4 — Nubes en transición: solo las nubes que realmente importan

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

# ── Panel 1: SCL original ─────────────────────────────────────────────────────
# Muestra el mapa SCL completo de la escena de prueba.
# Los colores van de negro (clase 0) a blanco (clase 9, nubes altas).
axes[0].imshow(scl_original, vmin=0, vmax=11)
axes[0].set_title(
    f"SCL original — {test_date}\n"
    f"{100*bad_fraction_original:.1f}% píxeles malos (filtro global)"
)
axes[0].axis("off")

# ── Panel 2: Reference Map ────────────────────────────────────────────────────
# Muestra el resultado del PASO 4: zonas estables y zona de transición.
# jet colormap: azul=0 (transición), verde=1 (agua), rojo=2 (tierra)
axes[1].imshow(reference_map, cmap="jet")
axes[1].set_title(
    "Reference Map\n"
    "azul=transición · verde=agua · rojo=tierra"
)
axes[1].axis("off")

# ── Panel 3: Zona intermareal ─────────────────────────────────────────────────
# La transition_mask en blanco/negro: blanco = zona de transición (value=True)
# Esta es la zona donde se evalúa la nubosidad en el filtro mejorado.
axes[2].imshow(transition_mask, cmap="gray")
axes[2].set_title(
    "Zona intermareal (transition_mask)\n"
    "blanco = zona de transición"
)
axes[2].axis("off")

# ── Panel 4: Nubes dentro de la zona de transición ────────────────────────────
# Intersección de (píxeles malos) AND (zona de transición).
# Estos son los únicos píxeles que importan para el filtrado mejorado.
clouds_transition = (
    np.isin(scl_original, scl_bad_classes)  # Píxeles malos en toda la escena
    & transition_mask                         # Solo los que están en la transición
)
axes[3].imshow(clouds_transition, cmap="Reds")
axes[3].set_title(
    f"Nubes EN la zona de transición\n"
    f"{stats_transition['bad_pct']:.1f}% (filtro mejorado)"
)
axes[3].axis("off")

plt.suptitle(
    f"Diagnóstico del filtro — {test_date}",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()


## 12 · Reference Map — construcción autónoma via OpenEO

A partir de aquí el análisis es independiente de las `selected_dates` del ejercicio anterior.

**Estrategia:**
1. Consultamos **todas las fechas disponibles** en el rango temporal completo.
2. Descargamos el **datacube SCL completo** (todas esas fechas, un solo batch job).
3. Calculamos `bad_fraction` para cada fecha directamente del datacube.
4. Seleccionamos las `reference_dates` con `bad_fraction ≤ umbral_referencia`.
5. Construimos el **reference map** en el backend (UDF + `reduce_dimension`) y lo descargamos como un **único GeoTIFF 2D** — en lugar de N SCLs individuales.


In [ ]:
# ── Importar la función de descarga del Reference Map ─────────────────────────
from utils import download_reference_map_openeo

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL REFERENCE MAP — año de referencia
# ════════════════════════════════════════════════════════════════════════════
#
# A partir de aquí el análisis es INDEPENDIENTE de selected_dates.
# Se usa un año completo (ref_time_extent) para construir el reference map
# con el máximo número de observaciones posibles → resultado más robusto.
#
# El año de referencia puede ser diferente al año de análisis (time_extent).
# Ideal: el mismo año o el año anterior al análisis.

# Año completo para construir el Reference Map
# (se descargará el cubo SCL completo en un solo batch job)
ref_time_extent = [
    "2020-01-01",
    "2024-12-31",
]

# Solo usar escenas con ≤5% de píxeles malos para el reference map
# (umbral más estricto que el filtro general del 20%)
ref_bad_fraction_threshold = 0.05

# Clases SCL consideradas malas en el reference map
# (se excluye la clase 11=nieve porque es rara en la costa gallega)
ref_bad_classes = [3, 8, 9, 10]

# Umbral de estabilidad: qué fracción de observaciones claras deben
# ser agua/tierra para clasificar el píxel como "estable"
# 0.95 → al menos 95% de las observaciones claras deben coincidir
ref_stable_threshold = 0.95

# Buffer espacial alrededor del límite agua/tierra (en píxeles a 20m/px)
# 10 píxeles = 200 metros de franja extra de transición
ref_coastal_buffer_pixels = 10

# Umbral para el filtro de nubes en la zona de transición (PASO 5)
# Una escena se acepta si tiene ≤20% de nubes dentro del estuario
# IMPORTANTE: Debe ser > ref_bad_fraction_threshold para recuperar fechas adicionales
transition_cloud_threshold = 0.1

# Ruta de salida para el GeoTIFF del reference map
ref_map_path = "reference_map_openeo.tif"

# Mostrar resumen de la configuración
print(f"Rango temporal reference map: {ref_time_extent[0]} → {ref_time_extent[1]}")
print(f"Umbral escenas limpias (reference): ≤{ref_bad_fraction_threshold*100:.0f}% píxeles malos")
print(f"Umbral estabilidad píxel:           ≥{ref_stable_threshold*100:.0f}% observaciones coincidentes")
print(f"Buffer costero:                      {ref_coastal_buffer_pixels} px ({ref_coastal_buffer_pixels*20} m)")
print(f"Umbral nubes en transición (PASO 5): ≤{transition_cloud_threshold*100:.0f}%")


In [ ]:
# ── Ejecutar el batch job del Reference Map en el backend ─────────────────────
# Esta celda lanza el job más costoso del pipeline: descarga todo el año de
# SCL (≈100 escenas) y ejecuta el UDF de votación directamente en el servidor.
#
# Tiempo estimado: 5-15 minutos dependiendo de la carga del backend.
#
# El resultado es un GeoTIFF de ~pocos MB con el reference map 2D.
# Si ref_map_path ya existe en disco, se puede cambiar force=False para saltarlo.
#
# Parámetros pasados al UDF:
#   - bad_fraction_threshold : escenas con >5% de nubes se ignoran en el voto
#   - stable_threshold       : umbral P_water para clasificar como estable
#   - transition_buffer_pixels: buffer espacial alrededor del límite agua/tierra

status_ref = download_reference_map_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=ref_time_extent,
    out_path=ref_map_path,
    bad_classes=ref_bad_classes,
    bad_fraction_threshold=ref_bad_fraction_threshold,
    stable_threshold=ref_stable_threshold,
    transition_buffer_pixels=ref_coastal_buffer_pixels,
    force=False,  # ← cambiar a False si el fichero ya existe y no quieres recalcular
)

print(f"Estado del job: {status_ref}")


In [ ]:
# ── Cargar el Reference Map descargado y visualizarlo ─────────────────────────
# Lee el GeoTIFF del reference map desde disco y extrae:
#   - reference_map_openeo : array 2D uint8 con valores 0/1/2
#   - ref_transform        : transformada afín (mapea píxeles a coordenadas)
#   - ref_crs              : sistema de referencia (normalmente UTM zona 29N)

from utils import load_reference_map_tif

reference_map_openeo, ref_transform, ref_crs = load_reference_map_tif(ref_map_path)

# La transition_mask es simplemente todos los píxeles con valor 0
# (aquellos que NO son ni agua estable ni tierra estable)
transition_mask = (reference_map_openeo == 0)

# ── Estadísticas del reference map ────────────────────────────────────────────
print(f"Dimensiones: {reference_map_openeo.shape}  ({reference_map_openeo.shape[1]*20}×{reference_map_openeo.shape[0]*20} m a 20m/px)")
print(f"CRS:         {ref_crs}")
print()

# Contar y mostrar la distribución de píxeles por clase
for val, label in [
    (0, "Transición   ← zona intermareal candidata"),
    (1, "Agua estable ← nunca al descubierto"),
    (2, "Tierra estable ← nunca inundada"),
]:
    n = (reference_map_openeo == val).sum()
    pct = 100 * n / reference_map_openeo.size
    print(f"  {val}  {label:<45}  {n:>8,} px ({pct:.1f}%)")

print(f"\n→ La zona de transición tiene {transition_mask.sum():,} píxeles")
print(f"  ({transition_mask.sum() * 400 / 1e6:.2f} km² a 20m/px)")

# Visualizar con colores: rojo=transición, azul=agua, arena=tierra
plot_reference_map(reference_map_openeo)


## 13 · Corrección SCL con Reference Map

Aplicamos el reference map a las fechas del rango completo que tengan `bad_fraction > umbral`.  
Los píxeles clasificados como sombra/nube sobre zonas estables conocidas se reasignan a agua (6) o tierra (5), reduciendo la fracción de malos.


### Re-autenticación OpenEO (en caso de error de conexión)

Si obtienes un error de conexión (`ConnectionResetError`, `ProtocolError`), ejecuta esta celda para re-autenticarte antes de reintentar.

In [ ]:
# ── Re-autenticación OpenEO (solo si hay errores de conexión) ────────────────
# Ejecutar esta celda si la siguiente falla con ConnectionResetError

try:
    # Verificar si ya está conectado
    account = conn.describe_account()
    print(f"✅ Ya conectado como: {account['name']}")
except:
    # Re-autenticar si falló
    print("🔄 Re-autenticando...")
    conn.authenticate_oidc()
    print(f"✅ Re-autenticado como: {conn.describe_account()['name']}")

In [ ]:
# ── Evaluación de nubosidad en zona de transición — PASO 5 ───────────────────
# Descarga el cubo SCL del año completo (ref_time_extent) como NetCDF y clasifica
# CADA fecha disponible en una de estas dos categorías:
#
#   FECHA DE REFERENCIA (automáticamente válida):
#     Si la fracción GLOBAL de píxeles malos ≤ ref_bad_fraction_threshold (5%)
#     → Esta fecha fue usada para construir el reference map. Se incluye en el
#       resultado con su fracción global real (que es ≤ 5%, siempre válida).
#
#   FECHA EVALUADA (filtro de transición):
#     Si la fracción global > 5% → Tiene nubes, pero pueden estar fuera del estuario.
#     → Se calcula el % de píxeles malos SOLO dentro de la zona de transición.
#     → Si ese % ≤ transition_cloud_threshold (20%) → fecha válida.
#
# Esta distinción es importante: las fechas de referencia son las mismas que
# el UDF usó internamente para votar agua/tierra en el reference map, y son
# válidas por definición para el water frequency.

from utils import evaluate_transition_cloud_coverage_openeo

transition_stats_raw = evaluate_transition_cloud_coverage_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=ref_time_extent,            # Año completo: "2023-01-01" → "2023-12-31"
    reference_map=reference_map_openeo,     # Reference map descargado en la sección 12
    reference_transform=ref_transform,      # Transformada afín del reference map
    bad_classes=ref_bad_classes,            # Clases SCL malas (mismas que usó el UDF)
    reference_dates=[],                     # No pre-excluimos: identificamos internamente
    global_bad_fraction_threshold=ref_bad_fraction_threshold,  # 5% = umbral del reference map
)

# Convertir al formato estándar del pipeline: {fecha: {"bad_fraction": float, "bad_pct": float}}
transition_stats = {
    date: {
        "bad_fraction": pct / 100,
        "bad_pct":      pct,
    }
    for date, pct in transition_stats_raw.items()
}

# Contar cuántas fechas fueron identificadas como "de referencia" (≤5% global)
n_ref_quality = sum(1 for s in transition_stats.values() if s["bad_fraction"] <= ref_bad_fraction_threshold)
n_transition_only = sum(1 for s in transition_stats.values() if s["bad_fraction"] > ref_bad_fraction_threshold and s["bad_fraction"] <= transition_cloud_threshold)
n_discarded = sum(1 for s in transition_stats.values() if s["bad_fraction"] > transition_cloud_threshold)

print(f"Fechas evaluadas en {ref_time_extent[0]} → {ref_time_extent[1]}: {len(transition_stats)}")
print()
print(f"  Calidad de referencia (≤{ref_bad_fraction_threshold*100:.0f}% global): {n_ref_quality:3d} ← usadas para construir el reference map")
print(f"  Válidas solo en transición:                    {n_transition_only:3d} ← estuario despejado pese a nubes en el AOI")
print(f"  Descartadas (>{transition_cloud_threshold*100:.0f}% en transición):        {n_discarded:3d}")
print()
for date, s in sorted(transition_stats.items()):
    if s["bad_fraction"] <= ref_bad_fraction_threshold:
        tag = "[ref]"
    elif s["bad_fraction"] <= transition_cloud_threshold:
        tag = "[✓]  "
    else:
        tag = "[✗]  "
    print(f"  {tag} {date}: {s['bad_pct']:.1f}%")


In [ ]:
# ── Clasificar fechas según el filtro de transición ──────────────────────────
# Aplicar el umbral transition_cloud_threshold para dividir las fechas
# evaluadas en válidas y descartadas.
#
# transition_stats viene de la celda anterior (evaluate_transition_cloud_coverage_openeo)
# Contiene TODAS las fechas del año completo (ref_time_extent), no solo las 8 de muestra.
# Formato: {fecha: {"bad_fraction": float, "bad_pct": float}}

# Fechas que pasan el filtro de transición
valid_dates_transition = [
    d
    for d, s in transition_stats.items()
    if s["bad_fraction"] <= transition_cloud_threshold
]

# Fechas que no pasan el filtro (demasiadas nubes sobre el estuario)
discarded_dates_transition = [
    d
    for d, s in transition_stats.items()
    if s["bad_fraction"] > transition_cloud_threshold
]

print(f"Umbral aplicado: ≤{transition_cloud_threshold*100:.0f}% nubes en zona de transición")
print()
print(f"Fechas válidas del año completo:  {len(valid_dates_transition):3d}")
print(f"Fechas descartadas:               {len(discarded_dates_transition):3d}")
print(f"─────────────────────────────────────────────")
print(f"TOTAL fechas para water frequency: {len(valid_dates_transition):3d}")


In [ ]:
# ── Fechas recuperadas gracias al filtro de transición ───────────────────────
# Compara cuántas fechas del año de referencia (2023) pasan el filtro de
# transición frente a cuántas hubieran pasado el filtro global estricto
# del reference map (≤5% global).
#
# IMPORTANTE: transition_stats contiene DOS tipos de valores mezclados:
#   - Para fechas ≤5% global: la fracción GLOBAL (ej: 3.2%)
#   - Para fechas >5% global: la fracción EN TRANSICIÓN (ej: 12.5%)
#
# Por eso separamos así:
#   - REFERENCIA: transition_stats ≤5% → son las que tienen ≤5% GLOBAL
#   - RECUPERADAS: transition_stats >5% pero ≤20% → tienen >5% global pero transición despejada

# Fechas de referencia: las que tienen ≤5% en transition_stats son las que
# tienen ≤5% GLOBAL (la función devuelve fracción global para estas)
reference_dates = sorted([
    d for d, s in transition_stats.items()
    if s["bad_fraction"] <= ref_bad_fraction_threshold
])

# Fechas recuperadas: las que tienen >5% en transition_stats (o sea, >5% global)
# pero ≤20% en transición (el valor que devuelve transition_stats para ellas)
newly_valid = sorted([
    d for d, s in transition_stats.items()
    if s["bad_fraction"] > ref_bad_fraction_threshold 
    and s["bad_fraction"] <= transition_cloud_threshold
])

print(f"Fechas del año {ref_time_extent[0][:4]} que pasan el filtro de transición: {len(valid_dates_transition)}")
print(f"  → Calidad de referencia (≤{ref_bad_fraction_threshold*100:.0f}% global):        {len(reference_dates):3d} fechas")
print(f"  → Recuperadas por filtro de transición:          {len(newly_valid):3d} fechas")
print(f"     (>{ref_bad_fraction_threshold*100:.0f}% global pero ≤{transition_cloud_threshold*100:.0f}% en transición)")
print()

if newly_valid:
    print(f"Primeras 10 fechas recuperadas por el filtro de transición:")
    for d in newly_valid[:10]:
        trans_pct = transition_stats[d]["bad_pct"]
        print(f"  {d}  [nubes en transición: {trans_pct:.1f}%]  (>5% global)")
    if len(newly_valid) > 10:
        print(f"  ... y {len(newly_valid) - 10} fechas más")
    print()
    print(f"✓ El filtro de transición recuperó {len(newly_valid)} fechas adicionales")
    print(f"  ({len(newly_valid)/len(valid_dates_transition)*100:.1f}% del total de fechas válidas)")
else:
    print("⚠️  No se recuperaron fechas adicionales.")
    print("   Esto significa que NO hay fechas con nubes fuera del estuario")
    print("   pero estuario despejado en este año.")
    if transition_cloud_threshold == ref_bad_fraction_threshold:
        print(f"\n   NOTA: transition_cloud_threshold ({transition_cloud_threshold*100:.0f}%) = ref_bad_fraction_threshold ({ref_bad_fraction_threshold*100:.0f}%)")
        print("   Considera aumentar transition_cloud_threshold a 0.20 para recuperar más fechas.")


In [ ]:
# ── Histograma temporal: distribución de fechas válidas a lo largo del año ──
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Convertir fechas a objetos datetime para el histograma
dates_ref = [datetime.strptime(d, "%Y-%m-%d") for d in reference_dates]
dates_new = [datetime.strptime(d, "%Y-%m-%d") for d in newly_valid]

# Crear figura
fig, ax = plt.subplots(figsize=(14, 5))

# Configurar bins mensuales para todo el año 2023
# Crear bins manualmente: primer día de cada mes + 1 de enero 2024
bins_dates = [datetime(2023, m, 1) for m in range(1, 13)] + [datetime(2024, 1, 1)]

# Histograma apilado (solo fechas válidas)
ax.hist(
    [dates_ref, dates_new],
    bins=bins_dates,
    label=[
        f'Referencia ≤5% global (n={len(dates_ref)})',
        f'Recuperadas >5% global, ≤20% transición (n={len(dates_new)})'
    ],
    color=['#5cb85c', '#5bc0de'],
    stacked=True,
    edgecolor='white',
    linewidth=0.5
)

# Formateo del eje X
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_xlabel('Mes (2023)', fontsize=11)
ax.set_ylabel('Número de escenas válidas', fontsize=11)
ax.set_title(
    f'Distribución temporal de escenas válidas - Ría de Foz ({ref_time_extent[0]} a {ref_time_extent[1]})',
    fontsize=12,
    pad=15
)
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Resumen estadístico por mes
print("\n── Cobertura mensual de escenas válidas ──")
valid_dates_dt = dates_ref + dates_new
months_count = {}
for dt in valid_dates_dt:
    month = dt.strftime("%Y-%m")
    months_count[month] = months_count.get(month, 0) + 1

for month in sorted(months_count.keys()):
    count = months_count[month]
    month_name = datetime.strptime(month, "%Y-%m").strftime("%B %Y")
    bar = "█" * count
    print(f"  {month_name:15s}  {count:2d} escenas  {bar}")

In [ ]:
# ── Cuantificar la mejora del filtro de transición sobre el filtro global ─────
# Compara cuántas fechas del AÑO COMPLETO (2023) pasan cada filtro:
#
#   Filtro global estricto (≤5%): fechas usadas para construir el reference map
#   Filtro de transición  (≤20%): fechas con estuario despejado aunque el AOI esté nublado
#
# Ambos valores salen de transition_stats, que ya los tiene separados por el
# parámetro global_bad_fraction_threshold que pasamos a evaluate_transition_cloud_coverage_openeo.

# Fechas con calidad de referencia: ≤5% píxeles malos en todo el AOI
n_ref_quality = sum(
    1 for s in transition_stats.values()
    if s["bad_fraction"] <= ref_bad_fraction_threshold
)

# Fechas recuperadas: >5% global pero ≤20% en la zona de transición
n_recovered_transition = sum(
    1 for s in transition_stats.values()
    if ref_bad_fraction_threshold < s["bad_fraction"] <= transition_cloud_threshold
)

n_total = len(transition_stats)

gain = quantify_reference_gain(
    n_initial_valid=n_ref_quality,         # Fechas que pasarían solo el filtro global (≤5%)
    n_recovered=n_recovered_transition,    # Fechas adicionales gracias al filtro de transición
    n_total_dates=n_total,                 # Total de fechas del año (2023)
)


## 14 · Water Frequency Raster

Para cada píxel del AOI se calcula la fracción de observaciones (solo escenas válidas) en que el SCL registró agua (clase 6), usando únicamente observaciones claras (SCL ∈ {4, 5, 6}):

$$\text{water\_frequency}(x, y) = \frac{\sum_t \mathbf{1}[\text{SCL}_t = 6]}{\sum_t \mathbf{1}[\text{SCL}_t \in \{4, 5, 6\}]}$$

- **≈ 0** → siempre tierra (por encima de la línea de pleamar)
- **≈ 1** → siempre agua (por debajo de la línea de bajamar)
- **0.10 – 0.95** → zona intertidal (intermitentemente inundada)

In [ ]:
# ── Cálculo del Water Frequency Raster — PASO 6 ──────────────────────────────
# Descarga el cubo SCL del año como NetCDF y calcula, para cada píxel:
#
#   water_frequency = votos_agua / observaciones_claras
#
# donde:
#   votos_agua          = fechas en que SCL == 6 (agua)
#   observaciones_claras = fechas en que SCL ∈ {4, 5, 6}
#                          (vegetación, suelo desnudo, agua → observación válida)
#
# Solo se incluyen las fechas de all_valid_dates (referencia + evaluación).
# Las fechas con nubes sobre el estuario quedan excluidas, evitando que
# los días nublados (donde no se ve el agua) distorsionen el ratio hacia 0.

from utils import compute_water_frequency_openeo, plot_water_frequency

# Unir todas las fechas válidas: las del reference map + las de la evaluación
# set() para eliminar duplicados; sorted() para ordenar cronológicamente
all_valid_dates = sorted(set(valid_dates_transition) | set(reference_dates))

print(f"Fechas válidas totales para water frequency: {len(all_valid_dates)}")
print(f"  → Del reference map (año 2023):  {len(reference_dates)}")
print(f"  → De la evaluación (excl. ref.): {len(valid_dates_transition)}")
print()

wf_out_path = "water_frequency.tif"

# Lanzar el cálculo (puede tardar varios minutos si force=True):
#   1. Descarga el cubo SCL del año ref_time_extent como NetCDF.
#   2. Filtra a all_valid_dates.
#   3. Calcula water_frequency píxel a píxel.
#   4. Guarda el resultado como GeoTIFF.
water_freq, wf_transform, wf_crs = compute_water_frequency_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=ref_time_extent,   # Año de referencia (donde están las fechas válidas)
    valid_dates=all_valid_dates,    # Solo estas fechas entran en el cálculo
    out_path=wf_out_path,
    force=False,  # ← True para recalcular aunque ya exista el fichero
)

print(f"Water frequency calculada:")
print(f"  Shape: {water_freq.shape}")
print(f"  CRS:   {wf_crs}")
print(f"  Valores: [{np.nanmin(water_freq):.3f}, {np.nanmax(water_freq):.3f}]")


In [ ]:
# ── Visualizar el Water Frequency raster ─────────────────────────────────────
# Muestra el mapa de frecuencia de agua sobre mapa base OpenStreetMap.
#
# Interpretación de los colores:
#   Blanco (#ffffff)    → 0.0  : nunca visto como agua → tierra firme
#   Azul claro          → ~0.3 : intermitente (zona intertidal alta)
#   Azul medio          → ~0.6 : inundación frecuente (zona intertidal baja)
#   Azul oscuro (#08306b)→ 1.0 : siempre agua → canal permanente
#
# El contorno amarillo resalta el umbral intertidal:
# el límite entre "frecuencia baja" y "frecuencia alta" que corresponde
# aproximadamente a la línea de bajamar media.

plot_water_frequency(
    water_freq,
    transform=wf_transform,   # Necesario para georreferenciar sobre el mapa base
    crs=wf_crs,               # CRS del raster (UTM zona 29N si se descargó correctamente)
    title="Frecuencia de agua — Ría de Foz",
)


In [ ]:
# ── Creación de la máscara intertidal final — PASO 7 ─────────────────────────
# Combina el water frequency raster (PASO 6) con el reference map (PASO 4)
# para obtener la máscara intertidal definitiva.
#
# La zona intertidal se define como los píxeles que cumplen TODAS las condiciones:
#   1. reference_map == 0   → están en la zona de transición
#   2. water_freq > low_thr → se ven como agua al menos "low_thr" del tiempo
#                              (descarta tierra firme que rozó el agua puntualmente)
#   3. water_freq < high_thr→ no se ven como agua siempre
#                              (descarta el canal permanente)
#
# Los umbrales estándar de FAST/OITC (Intertidal Science):
#   low  = 0.05  → 5% del tiempo bajo el agua = límite tierra/intertidal alta
#   high = 0.95  → 95% del tiempo bajo el agua = límite intertidal baja/canal

low_threshold  = 0.01   # Frecuencia mínima para ser considerado intertidal
high_threshold = 0.99   # Frecuencia máxima para no ser considerado canal

# La máscara intertidal final combina los tres criterios
intertidal_mask = (
    (reference_map_openeo == 0)          # Criterio 1: zona de transición
    & (water_freq > low_threshold)       # Criterio 2: inundado > 5% del tiempo
    & (water_freq < high_threshold)      # Criterio 3: no siempre inundado
)

# ── Estadísticas ──────────────────────────────────────────────────────────────
total_intertidal = intertidal_mask.sum()
area_km2 = total_intertidal * (20 * 20) / 1e6  # píxeles × (20m)² → km²

print(f"MÁSCARA INTERTIDAL — Ría de Foz 2023")
print(f"─────────────────────────────────────")
print(f"Zona de transición (reference map): {(reference_map_openeo==0).sum():>8,} px")
print(f"  → intertidal (0.05 ≤ WF ≤ 0.95): {total_intertidal:>8,} px  ({area_km2:.2f} km²)")
print(f"  → tierra (WF < 0.05):             {((reference_map_openeo==0) & (water_freq <= low_threshold)).sum():>8,} px")
print(f"  → canal (WF > 0.95):              {((reference_map_openeo==0) & (water_freq >= high_threshold)).sum():>8,} px")

# ── Guardar la máscara como GeoTIFF ──────────────────────────────────────────
# Si wf_transform y wf_crs están disponibles, el GeoTIFF estará georreferenciado.
# Si no (problema conocido con NetCDF), se guarda sin CRS.
intertidal_out_path = "intertidal_mask.tif"

if wf_transform is not None and wf_crs is not None:
    # Guardar con metadatos espaciales completos
    import rasterio
    from rasterio.transform import from_bounds
    height, width = intertidal_mask.shape
    with rasterio.open(
        intertidal_out_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=wf_crs,
        transform=wf_transform,
    ) as dst:
        dst.write(intertidal_mask.astype("uint8"), 1)
    print(f"\nGeoTIFF georreferenciado guardado en: {intertidal_out_path}")
else:
    # Guardar sin georreferenciación (problema conocido: rioxarray no extrae CRS del NetCDF)
    import rasterio
    height, width = intertidal_mask.shape
    with rasterio.open(
        intertidal_out_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
    ) as dst:
        dst.write(intertidal_mask.astype("uint8"), 1)
    print(f"\nGeoTIFF guardado SIN georreferenciación en: {intertidal_out_path}")
    print("  ⚠ Nota: wf_transform/wf_crs = None (problema conocido con rioxarray+NetCDF)")

# ── Visualización final ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(intertidal_mask, cmap="Blues")
ax.set_title(
    "Máscara intertidal final — Ría de Foz 2023\n"
    f"Umbral: WF ∈ [{low_threshold:.2f}, {high_threshold:.2f}]  |  "
    f"{area_km2:.2f} km²"
)
ax.axis("off")
plt.tight_layout()
plt.show()


## 15 · Fichas válidas finales y descarga RGB de validación

Resumen de todas las fechas que superan el pipeline completo. Se distinguen dos categorías:
- **[ref]** — calidad de referencia (≤5% píxeles malos en todo el AOI); usadas para construir el reference map.
- **[✓]** — recuperadas por el filtro de transición (>5% global pero ≤20% de nubes en la zona intermareal).

A continuación se descargan imágenes RGB de una muestra de las fechas **[✓]** para validarlas visualmente.

In [ ]:
# ── Lista de fichas válidas finales ──────────────────────────────────────────
# all_valid_dates contiene la unión de:
#   [ref] — fechas con ≤5% píxeles malos globales (calidad reference map)
#   [✓]   — fechas recuperadas por el filtro de transición
#            (>5% global pero ≤20% de nubes en la zona de transición)

# Identificar las fechas de cada categoría a partir de transition_stats
dates_ref_quality = sorted(
    d for d, s in transition_stats.items()
    if s["bad_fraction"] <= ref_bad_fraction_threshold
)
dates_transition_only = sorted(
    d for d, s in transition_stats.items()
    if ref_bad_fraction_threshold < s["bad_fraction"] <= transition_cloud_threshold
)

print("═" * 60)
print(f"  FICHAS VÁLIDAS FINALES — {site}  ({ref_time_extent[0][:4]})")
print("═" * 60)
print()
print(f"  [ref]  Calidad de referencia (≤{ref_bad_fraction_threshold*100:.0f}% global): {len(dates_ref_quality):3d} fechas")
for d in dates_ref_quality:
    pct = transition_stats[d]["bad_pct"]
    print(f"         {d}   {pct:.1f}% malos (global)")

print()
print(f"  [✓]   Recuperadas por transición "
      f"(>{ref_bad_fraction_threshold*100:.0f}% global, ≤{transition_cloud_threshold*100:.0f}% en estuario): "
      f"{len(dates_transition_only):3d} fechas")
for d in dates_transition_only:
    pct = transition_stats[d]["bad_pct"]
    print(f"         {d}   {pct:.1f}% nubes en transición")

# Combinar ambas listas en all_valid_dates
all_valid_dates = sorted(dates_ref_quality + dates_transition_only)

print()
print("─" * 60)
print(f"  TOTAL fichas válidas: {len(all_valid_dates):3d}")
print("═" * 60)

# ── Variable reutilizable para el modelo de mareas ────────────────────────────
# Lista ordenada de todas las fechas válidas del pipeline.
# Se puede usar directamente en análisis futuros (p. ej. serie temporal de mareas).
valid_scene_dates = all_valid_dates   # lista de strings ISO "YYYY-MM-DD"

print(f"\nGuardado como `valid_scene_dates` ({len(valid_scene_dates)} fechas)")


In [ ]:
# ── Descarga RGB para validación visual — fechas recuperadas por transición ───
# Las imágenes con corte diagonal son fechas en que el tile de Sentinel-2
# no cubría completamente el bbox (límite orbital). Se escanean TODAS las
# fechas de transición en busca de imágenes ya descargadas con buena cobertura,
# y si no hay suficientes se descarga una muestra mayor para tener dónde elegir.
#
# ► Ajusta n_download  : cuántas fechas descargar si faltan (None = todas).
# ► Ajusta min_coverage: umbral mínimo de cobertura del tile (0–1).
# ► Ajusta n_show      : cuántas imágenes mostrar en el grid final.

n_download   = 25  # ← fechas a descargar si no hay suficientes en disco
min_coverage = 0.6# ← descartar imágenes con <70% del bbox cubierto
n_show       = 4    # ← imágenes a mostrar en el grid

# ── Cobertura real del tile: fracción de píxeles con datos (no-negro) ─────────
def rgb_coverage(date, rgb_dir):
    path = os.path.join(rgb_dir, f"rgb_{date}.tif")
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as src:
        arr = src.read()
    return np.any(arr > 0, axis=0).mean()

# Paso 1: escanear fechas que ya están en disco
already_ok = {
    d: rgb_coverage(d, dir_rgb)
    for d in dates_transition_only
    if os.path.exists(os.path.join(dir_rgb, f"rgb_{d}.tif"))
}
good_on_disk = [d for d, cov in already_ok.items() if cov is not None and cov >= min_coverage]

print(f"Fechas [✓] con RGB en disco:          {len(already_ok)}")
print(f"  → cobertura ≥ {min_coverage*100:.0f}%:              {len(good_on_disk)}")

# Paso 2: descargar más si no tenemos suficientes
results_rgb_transition = {d: "skipped" for d in already_ok}

if len(good_on_disk) < n_show:
    pending = [d for d in dates_transition_only if d not in already_ok]
    to_dl   = pending[:n_download] if n_download else pending
    print(f"\nDescargando {len(to_dl)} fechas adicionales para buscar cobertura completa…\n")
    for date in to_dl:
        pct = transition_stats[date]["bad_pct"]
        print(f"  → {date}  [{pct:.1f}% nubes en transición]")
        status = download_date_rgb(conn, date, bbox, dir_rgb)
        results_rgb_transition[date] = status
        if status in ("ok", "skipped"):
            cov = rgb_coverage(date, dir_rgb)
            already_ok[date] = cov
            if cov is not None and cov >= min_coverage:
                good_on_disk.append(date)
                good_on_disk = sorted(set(good_on_disk))
                if len(good_on_disk) >= n_show:
                    print(f"  ✓ Ya tenemos {n_show} imágenes con cobertura suficiente — parando.")
                    break

# Paso 3: tabla de cobertura y grid
print("\n── Cobertura del tile ──")
for d in sorted(already_ok):
    cov  = already_ok[d]
    flag = "" if (cov is not None and cov >= min_coverage) else "  ← omitida (tile parcial)"
    if cov is not None:
        print(f"  {d}   {transition_stats[d]['bad_pct']:.1f}% nubes · {cov*100:.0f}% tile{flag}")
    else:
        print(f"  {d}   sin datos")

dates_show = sorted(good_on_disk)[:n_show]
print(f"\n  Mostrando ({len(dates_show)}): {dates_show}")

if dates_show:
    plot_rgb_grid(
        dates=dates_show,
        rgb_dir=dir_rgb,
        polygon=polygon,
        title=f"Validación visual — fechas [✓] recuperadas por transición — {site}  (tile ≥ {min_coverage*100:.0f}%)",
        cols=min(4, len(dates_show)),
    )
else:
    print(f"Ninguna imagen supera el umbral de cobertura ({min_coverage*100:.0f}%). Reduce min_coverage o aumenta n_download.")


In [ ]:
pip install -r requirements.txt

## 16 · Análisis de altura de marea para fechas válidas

Este análisis correlaciona las fechas válidas con su altura de marea correspondiente, permitiendo:
- Identificar si la cobertura temporal incluye mareas altas y bajas.
- Validar que la serie cubre un rango completo de estados mareales (esencial para el análisis intermareal).
- Detectar sesgos sistemáticos hacia ciertos rangos de marea que podrían afectar el cálculo de water frequency.

Usamos el modelo de mareas **GOT4.10** (Global Ocean Tide) vía [pyTMD](https://github.com/tsutterley/pyTMD) para estimar la altura de marea en el centroide del AOI para cada fecha válida.

In [ ]:
import importlib
import tidemodel
importlib.reload(tidemodel)
from tidemodel import PyTMDTideModel

# Recrear el modelo con los parámetros correctos
tide_model = PyTMDTideModel(
    model_name="GOT4.10", 
    directory="./tide_models",
    box_size=0.4,
    resolution=0.05
)

In [ ]:
# ── Calcular altura de marea para todas las fechas válidas ──────────────────
from tidemodel import PyTMDTideModel
from datetime import datetime
from utils import get_water_centroid

# Calcular centroide del AOI (asumiendo que cae en zona de agua/transición)
# Si se dispone del reference_map, se puede usar get_water_centroid para
# asegurar que el punto cae en agua.
lon_centroid, lat_centroid = get_water_centroid(polygon)

print(f"Centroide del AOI: {lat_centroid:.6f}°N, {lon_centroid:.6f}°W")

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN: Limitar número de fechas para pruebas rápidas
# ════════════════════════════════════════════════════════════════════════════
# Opciones:
#   None  → Procesar todas las fechas (puede tardar 10-15 minutos con ~176 fechas)
#   14    → Últimas 2 semanas (~10 fechas, tarda ~1-2 minutos)
#   30    → Último mes (~20 fechas, tarda ~3-5 minutos)

LIMIT_DAYS = None  # Cambiar a None para procesar todas las fechas

if LIMIT_DAYS is not None:
    from datetime import timedelta
    # Filtrar solo fechas recientes
    max_date = max(all_valid_dates)
    cutoff_date = (datetime.strptime(max_date, "%Y-%m-%d") - timedelta(days=LIMIT_DAYS)).strftime("%Y-%m-%d")
    dates_to_process = [d for d in all_valid_dates if d >= cutoff_date]
    print(f"⚠️  MODO PRUEBA: procesando últimos {LIMIT_DAYS} días ({len(dates_to_process)} fechas de {len(all_valid_dates)} totales)")
    print(f"   Rango: {min(dates_to_process)} → {max(dates_to_process)}")
    print(f"   Para procesar todo, cambiar LIMIT_DAYS = None")
else:
    dates_to_process = all_valid_dates
    print(f"Calculando mareas para {len(dates_to_process)} fechas válidas (TODAS)...")

# Inicializar modelo de mareas GOT4.10
# box_size: tamaño de la caja de búsqueda (±2.0° ~ 222 km, amplio para asegurar datos válidos)
# resolution: resolución de muestreo (0.1° ~ 11 km)
tide_model = PyTMDTideModel(
    model_name="GOT4.10", 
    directory="./tide_models",
    box_size=2.0,
    resolution=0.1
)

print(f"Coordenadas para GOT4.10: {lat_centroid:.6f}°N, {lon_centroid:.6f}°W")
print(f"  (pyTMD usa formato WGS84 -180/180, sin conversión)")

# Calcular altura de marea para cada fecha válida
# Sentinel-2 pasa aproximadamente a las 10:30-11:00 UTC sobre Europa
# Usamos 11:00 UTC como hora estándar de adquisición
import time
tide_heights = []
start_time = time.time()

print(f"\n⏳ Procesando {len(dates_to_process)} fechas...")
for i, date_str in enumerate(dates_to_process, 1):
    # Convertir fecha YYYY-MM-DD a datetime a las 11:00 UTC
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(hour=11, minute=0)
    
    # Calcular altura de marea en metros
    # El modelo busca el punto válido (no-NaN) más cercano en una malla alrededor de (lat, lon)
    h = tide_model.get_tide_height(lat_centroid, lon_centroid, dt)
    tide_heights.append(h)
    
    # Mostrar progreso cada 10 fechas
    if i % 10 == 0 or i == len(dates_to_process):
        elapsed = time.time() - start_time
        rate = i / elapsed
        remaining = (len(dates_to_process) - i) / rate if rate > 0 else 0
        print(f"   {i}/{len(dates_to_process)} ({i/len(dates_to_process)*100:.1f}%) - "
              f"Velocidad: {rate:.1f} fechas/s - "
              f"Restante: {remaining:.0f}s")

elapsed_total = time.time() - start_time
print(f"\n✅ Completado en {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")

# Crear DataFrame para facilitar el análisis
import pandas as pd
df_tides = pd.DataFrame({
    "date": pd.to_datetime(dates_to_process),
    "tide_height_m": tide_heights
})

# Estadísticas descriptivas
print("\n── Estadísticas de altura de marea ──")
print(f"  Mínima:  {df_tides['tide_height_m'].min():.2f} m")
print(f"  Máxima:  {df_tides['tide_height_m'].max():.2f} m")
print(f"  Media:   {df_tides['tide_height_m'].mean():.2f} m")
print(f"  Rango:   {df_tides['tide_height_m'].max() - df_tides['tide_height_m'].min():.2f} m")
print(f"\nDatos guardados en `df_tides` (DataFrame con {len(df_tides)} filas)")

if LIMIT_DAYS is not None:
    print(f"\n⚠️  ATENCIÓN: Solo se procesaron {len(dates_to_process)} fechas de {len(all_valid_dates)} totales")
    print(f"   Para análisis completo, cambiar LIMIT_DAYS = None y re-ejecutar")

# Asignar también a valid_scene_dates para compatibilidad
valid_scene_dates = dates_to_process

### Comparación completa: GOT4.10 vs CMEMS para todas las fechas

Calculamos mareas en **un único punto** (centroide del AOI) para **todas las fechas válidas** con ambos modelos y comparamos resultados.

In [ ]:
# ── Cálculo de mareas completo: GOT4.10 vs CMEMS (todas las fechas) ─────────
import time
import numpy as np
import pandas as pd
from datetime import datetime
from tidemodel import PyTMDTideModel, CopernicusTideModel

print("COMPARACIÓN COMPLETA: GOT4.10 vs CMEMS")
print("="*70)
print(f"\n📍 Punto de cálculo: {lat_centroid:.4f}°N, {abs(lon_centroid):.4f}°W")
print(f"📅 Fechas a procesar: {len(all_valid_dates)} ({all_valid_dates[0]} → {all_valid_dates[-1]})")
print(f"🕐 Hora de adquisición Sentinel-2: 11:00 UTC")

# Inicializar modelos
print(f"\n⏳ Inicializando modelos...")
tide_model_got = PyTMDTideModel(
    model_name="GOT4.10", 
    directory="./tide_models",
    box_size=2.0,
    resolution=0.1
)
tide_model_cmems = CopernicusTideModel()

# Arrays para resultados
tide_got = []
tide_cmems = []
dates_processed = []

print(f"\n⏳ Calculando mareas para {len(all_valid_dates)} fechas...")
print(f"   (Esto puede tardar varios minutos)")

start_time = time.time()
n_errors_got = 0
n_errors_cmems = 0

for i, date_str in enumerate(all_valid_dates, 1):
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(hour=11, minute=0)
    
    # GOT4.10
    try:
        h_got = tide_model_got.get_tide_height(lat_centroid, lon_centroid, dt)
        tide_got.append(h_got)
    except Exception as e:
        tide_got.append(np.nan)
        n_errors_got += 1
    
    # CMEMS
    try:
        h_cmems = tide_model_cmems.get_tide_height(lat_centroid, lon_centroid, dt)
        tide_cmems.append(h_cmems)
    except Exception as e:
        tide_cmems.append(np.nan)
        n_errors_cmems += 1
    
    dates_processed.append(date_str)
    
    # Progreso cada 20 fechas
    if i % 20 == 0 or i == len(all_valid_dates):
        elapsed = time.time() - start_time
        rate = i / elapsed
        remaining = (len(all_valid_dates) - i) / rate if rate > 0 else 0
        print(f"   {i}/{len(all_valid_dates)} ({i/len(all_valid_dates)*100:.1f}%) - "
              f"Velocidad: {rate:.1f} fechas/s - "
              f"Restante: {remaining/60:.1f} min", end='\r')

elapsed_total = time.time() - start_time
print(f"\n\n✅ Completado en {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")

if n_errors_got > 0:
    print(f"⚠️  GOT4.10: {n_errors_got} errores de {len(all_valid_dates)} fechas")
if n_errors_cmems > 0:
    print(f"⚠️  CMEMS: {n_errors_cmems} errores de {len(all_valid_dates)} fechas")

# Crear DataFrame con resultados
df_comparison = pd.DataFrame({
    'date': dates_processed,
    'tide_got4_m': tide_got,
    'tide_cmems_m': tide_cmems
})
df_comparison['date'] = pd.to_datetime(df_comparison['date'])

# Filtrar datos válidos
got_valid = df_comparison['tide_got4_m'].dropna()
cmems_valid = df_comparison['tide_cmems_m'].dropna()

print(f"\n{'='*70}")
print(f"📊 ESTADÍSTICAS COMPARATIVAS")
print(f"{'='*70}")

print(f"\n🌊 GOT4.10 (modelo astronómico puro):")
print(f"   Datos válidos: {len(got_valid)}/{len(all_valid_dates)}")
print(f"   Mínima:  {got_valid.min():.3f} m")
print(f"   Máxima:  {got_valid.max():.3f} m")
print(f"   Media:   {got_valid.mean():.3f} m")
print(f"   Mediana: {got_valid.median():.3f} m")
print(f"   Desv.St: {got_valid.std():.3f} m")
print(f"   ⭐ Rango: {got_valid.max() - got_valid.min():.3f} m")

print(f"\n🌊 CMEMS (astronómico + meteorológico + circulación):")
print(f"   Datos válidos: {len(cmems_valid)}/{len(all_valid_dates)}")
print(f"   Mínima:  {cmems_valid.min():.3f} m")
print(f"   Máxima:  {cmems_valid.max():.3f} m")
print(f"   Media:   {cmems_valid.mean():.3f} m")
print(f"   Mediana: {cmems_valid.median():.3f} m")
print(f"   Desv.St: {cmems_valid.std():.3f} m")
print(f"   ⭐ Rango: {cmems_valid.max() - cmems_valid.min():.3f} m")

# Correlación entre modelos (solo fechas con ambos datos)
df_both = df_comparison.dropna(subset=['tide_got4_m', 'tide_cmems_m'])
if len(df_both) > 10:
    correlation = df_both['tide_got4_m'].corr(df_both['tide_cmems_m'])
    print(f"\n📈 Correlación GOT4.10 vs CMEMS: {correlation:.3f}")
    print(f"   (Basado en {len(df_both)} fechas con ambos datos)")

print(f"\n💡 DIAGNÓSTICO:")
got_range = got_valid.max() - got_valid.min()
cmems_range = cmems_valid.max() - cmems_valid.min()

if got_range > 3.0 and cmems_range > 3.0:
    print(f"   ✅ Ambos modelos muestran rango >3 m → FUNCIONAN CORRECTAMENTE")
    print(f"   ✅ Los modelos son válidos para esta ubicación")
elif got_range < 1.0 and cmems_range < 1.0:
    print(f"   ❌ Ambos modelos muestran rango <1 m → PROBLEMA SISTEMÁTICO")
    print(f"   → Posible causa: punto en zona sin cobertura de datos mareales")
    print(f"   → Solución: mover cálculo a punto más offshore")
elif cmems_range > 3.0 and got_range < 1.0:
    print(f"   ⚠️  CMEMS correcto pero GOT4.10 incorrecto")
    print(f"   → Problema específico con GOT4.10 en esta ubicación")
else:
    print(f"   ⚠️  Resultados inconsistentes entre modelos")
    print(f"   → Requiere investigación adicional")

print(f"\n{'='*70}")
print(f"Datos guardados en `df_comparison` (DataFrame con {len(df_comparison)} filas)")

In [ ]:
# ── Visualización: serie temporal GOT4.10 vs CMEMS ───────────────────────────
import plotly.graph_objects as go

df_plot = df_comparison.dropna(subset=['tide_got4_m', 'tide_cmems_m'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_plot['date'], y=df_plot['tide_got4_m'], mode='markers',
    name='GOT4.10 (astronómico)', marker=dict(size=7, color='blue', opacity=0.6),
    hovertemplate='<b>GOT4.10</b><br>%{x|%Y-%m-%d}<br>%{y:.2f} m<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df_plot['date'], y=df_plot['tide_cmems_m'], mode='markers',
    name='CMEMS (completo)', marker=dict(size=7, color='red', opacity=0.6),
    hovertemplate='<b>CMEMS</b><br>%{x|%Y-%m-%d}<br>%{y:.2f} m<extra></extra>'
))
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="Datum GOT4.10", annotation_position="right")
fig.update_layout(
    title=f"Mareas en {site} | Rango GOT={got_range:.1f}m, CMEMS={cmems_range:.1f}m | r={correlation:.2f}",
    xaxis_title="Fecha", yaxis_title="Altura de marea (m)", height=500,
    hovermode='x unified', template='plotly_white',
    legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center")
)
fig.update_xaxes(rangeslider_visible=True)
fig.show()

## Validación con Mareógrafo: Explicación del Offset de Datum

### ¿Por qué hay un "offset" entre modelos y mareógrafo?

Cuando comparas GOT4.10 o CMEMS con datos de un mareógrafo de Puertos del Estado, verás un **desplazamiento vertical constante** (~0.3–3.0 m). Esto NO es un error del modelo, sino una **diferencia de datum** (nivel de referencia):

| Fuente | Datum de referencia | Valor típico |
|--------|---------------------|--------------|
| **Mareógrafo REDMAR** | Cero del puerto (local, arbitrario) | ~3.0 m |
| **GOT4.10** | Nivel medio del mar (geofísico global) | ~0.0 m |
| **CMEMS** | Nivel medio del mar (físico, incluye meteorología) | ~0.0 m |

**Ejemplo real (Santander, enero 2024):**
- Mareógrafo: nivel medio = **2.96 m** sobre cero local
- GOT4.10: nivel medio ≈ **0 m** (referencia global)
- Diferencia: **~3 m de offset**, pero **misma amplitud** (~4 m de carrera mareal)

### Solución: Trabajar con Anomalías

Para validar correctamente, calculamos **anomalías** (desviaciones del nivel medio de cada serie):

```python
anomalía = valor - media_de_la_serie
```

Esto elimina el offset y permite comparar:
- **Amplitud de marea** (rango pico-valle)
- **Fase temporal** (sincronización pleamar/bajamar)
- **Variabilidad** (patrones de ciclo mareal)

### 📍 Punto de Cálculo: Offshore vs Intermareal

Dentro de la ría, la batimetría compleja puede causar que los modelos globales subestimen la marea. Por eso:

- `USE_OFFSHORE = True`: calcula ~5 km mar adentro (aguas abiertas, modelo más preciso)
- `USE_OFFSHORE = False`: usa ubicación del mareógrafo (comparación directa, pero puede haber error batimétrico)

**Recomendación:** usar offshore para análisis, mareógrafo solo para validación.


In [ ]:
# ── Cargar mareógrafo PORTUS/REDMAR (xlsx diario o csv horario) ─────────────
import pandas as pd
import os

# Busca automáticamente cualquier archivo de mareógrafo en el directorio actual.
# PORTUS permite exportar en dos formatos:
#   - Diario  (.xlsx): estadísticos por día (Nivel Medio, Pleamar, Bajamar, Carrera)
#   - Horario (.csv): una medición por hora (Fecha, Hora, Valor)
gauge_files = [
    f for f in os.listdir(".")
    if f.lower().startswith("mareografo_") and f.lower().endswith((".xlsx", ".csv"))
]

if not gauge_files:
    print("❌ No hay archivo mareografo_*.xlsx/.csv en el directorio de trabajo.")
    print("   Descárgalo en: https://portus.puertos.es → Datos Históricos → Nivel del Mar")
    print("   Guárdalo con el nombre mareografo_<estacion>.xlsx  (o .csv si es horario)")
    df_mareografo = None
    obs_available = False
else:
    fpath = gauge_files[0]
    print(f"✓ Cargando {fpath}")

    # Leer el archivo según su extensión
    raw = (
        pd.read_excel(fpath) if fpath.endswith(".xlsx")
        else pd.read_csv(fpath, sep=";", decimal=",", encoding="latin1")
    )
    print(f"  {len(raw)} filas · columnas: {list(raw.columns)}")

    # ── Formato DIARIO (columna 'Fecha (GMT)') ────────────────────────────────
    # PORTUS exporta estadísticos diarios: nivel medio, pleamar, bajamar y carrera.
    # Los valores vienen en centímetros sobre el cero del puerto (datum local),
    # así que los convertimos a metros.
    if "Fecha (GMT)" in raw.columns:
        gauge_type = "diario"
        df_mareografo = pd.DataFrame({
            "date":          pd.to_datetime(raw["Fecha (GMT)"]),
            "nivel_medio_m": pd.to_numeric(raw["Nivel Medio (cm)"],  errors="coerce") / 100,
            "pleamar_m":     pd.to_numeric(raw["Pleamar Max. (cm)"], errors="coerce") / 100,
            "bajamar_m":     pd.to_numeric(raw["Baja Mar Min. (cm)"],errors="coerce") / 100,
            "carrera_m":     pd.to_numeric(raw["Carrera Max. (cm)"], errors="coerce") / 100,
        }).dropna(subset=["date", "nivel_medio_m"])

        print(f"  Formato: DIARIO | Periodo: {df_mareografo['date'].min().date()} → {df_mareografo['date'].max().date()}")
        print(f"  Nivel medio = {df_mareografo['nivel_medio_m'].mean():.2f} m sobre cero local")
        print(f"  Carrera media = {df_mareografo['carrera_m'].mean():.2f} m | máxima = {df_mareografo['carrera_m'].max():.2f} m")

    # ── Formato HORARIO (columnas 'Fecha', 'Hora', 'Valor') ──────────────────
    # PORTUS exporta una medición por hora. Para comparar con Sentinel-2 (paso
    # a las ~11:00 UTC) extraemos solo la medición de las 11:00 de cada día.
    elif "Fecha" in raw.columns and "Hora" in raw.columns:
        gauge_type = "horario"
        raw["datetime"] = pd.to_datetime(
            raw["Fecha"] + " " + raw["Hora"], format="%d/%m/%Y %H:%M", errors="coerce"
        )
        h11 = raw.dropna(subset=["datetime"]).query("datetime.dt.hour == 11").copy()
        df_mareografo = pd.DataFrame({
            "date":               pd.to_datetime(h11["datetime"].dt.date),
            "observed_height_m":  h11["Valor"].astype(float) / 100,  # cm → m
        }).dropna()
        print(f"  Formato: HORARIO | {len(df_mareografo)} mediciones a las 11:00 UTC")

    else:
        raise ValueError(
            f"Formato no reconocido. Columnas encontradas: {list(raw.columns)}\n"
            "Se esperan columnas 'Fecha (GMT)' (diario) o 'Fecha'+'Hora'+'Valor' (horario)."
        )

    obs_available = True
